# Пример получения сгенерированных изображений и видео в GigaChat
В этом примере мы разберем, как получать сгенерированные изображения и видео в GigaChat с помощью GigaChain

## Установка зависимостей

Для работы с примером установите зависимости:

In [ ]:
!pip install langchain-gigachat python-dotenv -q

## Инициализация GigaChat

Для работы с моделями GigaChat, нужно передать ключ авторизации GigaChat API.

Ключ можно указать в переменной среды `GIGACHAT_CREDENTIALS`, заданной в файле `.env` или созданной с помощью команды:

```sh
export GIGACHAT_CREDENTIALS=ключ_авториазации
```

При инициализации проверяется наличие переменной среды `GIGACHAT_CREDENTIALS` с заданным ключом авторизации GigaChat API.
Если переменная отсутствует, вы сможете указать ключ в поле **Введите ключ авторизации GigaChat API**.

О способах авторизации и поддерживаемых переменных среды — в [README библиотеки gigachat](https://github.com/ai-forever/gigachat).

In [ ]:
import getpass
import os
from dotenv import find_dotenv, load_dotenv
from langchain_gigachat.chat_models import GigaChat

load_dotenv(find_dotenv())

if "GIGACHAT_CREDENTIALS" not in os.environ:
    os.environ["GIGACHAT_CREDENTIALS"] = getpass.getpass("Введите ключ авторизации GigaChat API: ")


llm = GigaChat(
    model="GigaChat-Pro",
    timeout=6000,
    verify_ssl_certs=False,
)

Вызовите метод `bind_tools(tools=[], tool_choice="auto")`, чтобы сообщить модели о необходимости использовать встроенные функции.

In [22]:
# Без вызова этого метода GigaChat, не сможет генерировать изображения
llm = llm.bind_tools(tools=[], tool_choice="auto")

Подготавливаем цепочку для получения сгенерированных изображений

In [23]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Ты — Василий Кандинский.
На каждый запрос ты должен картину в стиле экспрессионизма.""",
        ),
        MessagesPlaceholder("history", optional=True),
        ("user", """{topic}"""),
    ]
)

generate_image_chain = prompt | llm

Пример получения изображения

In [ ]:
from base64 import b64decode

from IPython import display

response = generate_image_chain.invoke({"topic": "Нарисуй картину красного кота"})
image_uuid = response.additional_kwargs.get("image_uuid")
image = llm.get_file_content(image_uuid)
description = response.additional_kwargs["postfix_message"]
display.HTML(
    f'<img src="data:image/png;base64,{image.content}" width="300" /><br><p>{description}</p>'
)

Пример генерации изображений на контексте прошлых сообщений

In [ ]:
response = generate_image_chain.invoke(
    {"topic": "Теперь сделай его синим", "history": [response]}
)
image_uuid = response.additional_kwargs.get("image_uuid")
description = response.additional_kwargs["postfix_message"]
display.HTML(
    f'<img src="data:image/png;base64,{llm.get_file_content(image_uuid).content}" width="300" /><br><p>{description}</p>'
)

## Получение изображений в режиме стриминга

In [33]:
image_uuid = None
chunks = []
for chunk in generate_image_chain.stream({"topic": "Нарисуй картину розового кота"}):
    chunks.append(chunk)
    if chunk.type == "FunctionInProgressMessage":
        continue
    if chunk.additional_kwargs.get("image_uuid"):
        image_uuid = chunk.additional_kwargs.get("image_uuid")
    else:
        print(chunk.content, end="")

 нарисовал картину розового кота.

In [ ]:
display.HTML(
    f'<img src="data:image/png;base64,{llm.get_file_content(image_uuid).content}" width="300" />'
)

## Генерация видео
На данный момент генерация видео через GigaChat недоступно, но в предыдущих реализациях, можно было получать видео таким образом

In [39]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Ты — Василий Кандинский.
На каждый запрос ты должен генерировать видео.""",
        ),
        MessagesPlaceholder("history", optional=True),
        ("user", """{topic}"""),
    ]
)
generate_video_chain = prompt | llm

In [ ]:
video_uuid = None
cover_uuid = None
for chunk in generate_video_chain.stream({"topic": "полет в космосе"}):
    if chunk.type == "FunctionInProgressMessage":
        continue
    if image := chunk.additional_kwargs.get("video_uuid"):
        video_uuid = chunk.additional_kwargs.get("video_uuid")
        cover_uuid = chunk.additional_kwargs.get("cover_uuid")
    else:
        print(chunk.content, end="")

Видео, которое получилось можно посмотреть по этой ссылке (на github не работает отображение видео)
https://github.com/ai-forever/gigachain/tree/dev/cookbook/images_and_videos/video.mp4

И какая обложка

In [ ]:
display.HTML(
    f'<img src="data:image/png;base64,{llm.get_file_content(cover_uuid).content}" width="300" /><br>'
)